In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## LOADING THE DATA

In [8]:
movies = pd.read_csv('data/TMDB_230000_movies.csv')
movies.head()

,title,genres,overview,cast,director,producers,writers,poster_path,year_norm
0,Ariel,"Comedy, Drama, Romance, Crime",A Finnish man goes to the city to find a job a...,"Tarja Keinänen, Marja Packalén, Kari Helaseppä...",Aki Kaurismäki,Aki Kaurismäki,Aki Kaurismäki,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,0.681481
1,Shadows in Paradise,"Comedy, Drama, Romance","Nikander, a rubbish collector and would-be ent...","Ari Korhonen, Mari Rantasila, Erkki Rissanen, ...",Aki Kaurismäki,Mika Kaurismäki,Aki Kaurismäki,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,0.666667
2,Four Rooms,Comedy,It's Ted the Bellhop's first night on the job....,"Marisa Tomei, Kimberly Blair, Madonna, Tamlyn ...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Lawrence Bender, Quentin Tarantino, Alexandre ...","Quentin Tarantino, Robert Rodriguez, Alexandre...",/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg,0.733333
3,Judgment Night,"Action, Crime, Thriller","Four young friends, while taking a shortcut en...","Doug Wert, Angela Alvarado, Everlast, Sean O'G...",Stephen Hopkins,"Marilyn Vance, Gene Levy, Lloyd Segan","Lewis Colick, Jere Cunningham",/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg,0.718519
4,Star Wars,"Adventure, Action, Science Fiction",Princess Leia is captured and held hostage by ...,"Jeremy Sinden, Michael Leader, Derek Lyons, Al...",George Lucas,"Gary Kurtz, George Lucas",George Lucas,/6FfCtAuVAW8XJjZ7eWeLibRLWTw.jpg,0.600000


## EXPLORING DATA

In [9]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237898 entries, 0 to 237897
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   title        237898 non-null  object 
 1   genres       237898 non-null  object 
 2   overview     237898 non-null  object 
 3   cast         237898 non-null  object 
 4   director     237898 non-null  object 
 5   producers    237898 non-null  object 
 6   writers      237898 non-null  object 
 7   poster_path  237898 non-null  object 
 8   year_norm    237898 non-null  float64
dtypes: float64(1), object(8)
memory usage: 16.3+ MB


In [10]:
movies.isnull().sum()

title          0
genres         0
overview       0
cast           0
director       0
producers      0
writers        0
poster_path    0
year_norm      0
dtype: int64

## CONTENT-BASED RECOMMENDATION

This one looks at the features of the movies themselves — genre, director, cast, keywords. If you liked Inception, it finds other movies with similar characteristics. Similarity will be computed using Cosine Similarity on a TF-IDF matrix built from the metadata.

In [11]:
movies['features'] = movies['genres'] + ' ' + movies['overview'] + ' ' + movies['cast'] + ' ' + movies['director'] + ' ' + movies['producers'] + ' ' + movies['writers'] 

🎯 TF‑IDF = “How important is this word for describing this movie?”
TF‑IDF stands for:

TF — Term Frequency  
How often a word appears in a document (movie description).

IDF — Inverse Document Frequency  
How rare that word is across the entire dataset.

Then it multiplies them:

TF-IDF
=
TF
×
IDF
🧠 Why this matters for your recommender
TF‑IDF gives high weight to words that are:

frequent in this movie

rare in other movies

And low weight to words that are:

common everywhere (“the”, “movie”, “story”, “man”, “life”)

movies x corpus
cell values= TFXIDF score of a word in this movie

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=50000  # optional, keeps it manageable
)

tfidf_matrix = tfidf.fit_transform(movies['features'])

tfidf_matrix


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12411453 stored elements and shape (237898, 50000)>

In [13]:
from sklearn.metrics.pairwise import linear_kernel

# Precomputing the full matrix leads to a memory crash (237k x 237k)
# Thats why we compute similarity on-the-fly per movie
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [14]:
def get_recommendations(title, top_n=10):
    idx = indices[title]
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_scores[idx] = 0
    top_indices = sim_scores.argsort()[::-1][:top_n]
    return movies['title'].iloc[top_indices]

In [15]:
get_recommendations("Inception")

92953                                Dunkirk
54296                           Interstellar
213044                             Promcakes
24063                  The Dark Knight Rises
154904                            Tarantella
21340                              Doodlebug
5936                               Following
106915                               Larceny
824                             The Prestige
86035     Dreams: Cinema of the Subconscious
Name: title, dtype: object

## COLLABORATIVE FILTERING

In [ ]:
import pandas as pd
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate
from collections import defaultdict

ratings = pd.read_csv('data/ratings.csv')       # userId, movieId, rating, timestamp
ml_movies = pd.read_csv('data/movies.csv')       # movieId, title, genres

print(f"Ratings: {len(ratings):,} rows")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['movieId'].nunique():,}")
ratings.head()

### Training the SVD Model

SVD (Singular Value Decomposition) decomposes the user-movie rating matrix into latent factors.  
It learns hidden patterns like *"this user likes action movies with complex plots"* without being told explicitly.

In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

svd_model = SVD(n_factors=50, n_epochs=20, random_state=42)
cv_results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

In [ ]:
import numpy as np

print(f"Mean RMSE: {np.mean(cv_results['test_rmse']):.4f}")
print(f"Mean MAE:  {np.mean(cv_results['test_mae']):.4f}")

In [ ]:
trainset = data.build_full_trainset()
svd_model.fit(trainset)
print("Model trained on full dataset.")

In [ ]:
def get_cf_recommendations(user_id, n=10):
    """
    Returns top-N movie recommendations for a given user_id.
    Excludes movies the user has already rated.
    """
    # Movies this user has already seen
    watched = set(ratings[ratings['userId'] == user_id]['movieId'])
    
    # All movies in the dataset
    all_movie_ids = ratings['movieId'].unique()
    
    # Predict rating for every unseen movie
    predictions = []
    for movie_id in all_movie_ids:
        if movie_id not in watched:
            pred = svd_model.predict(user_id, movie_id)
            predictions.append((movie_id, pred.est))
    
    # Sort by predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_ids = [p[0] for p in predictions[:n]]
    
    # Merge with movie titles
    result = ml_movies[ml_movies['movieId'].isin(top_ids)][['movieId', 'title', 'genres']].copy()
    result['predicted_rating'] = result['movieId'].map(dict(predictions[:n]))
    result = result.sort_values('predicted_rating', ascending=False).reset_index(drop=True)
    return result

In [ ]:
get_cf_recommendations(user_id=1, n=10)

### Comparing Content-Based vs Collaborative Filtering

Let's compare recommendations for the same movie to see how the two approaches differ.

In [ ]:
import matplotlib.pyplot as plt

# Pick a user and see what they rated highly
user_id = 1
user_ratings = ratings[ratings['userId'] == user_id].merge(ml_movies, on='movieId')
top_rated = user_ratings.sort_values('rating', ascending=False).head(5)[['title', 'genres', 'rating']]

print(f"User {user_id}'s top-rated movies:")
print(top_rated.to_string(index=False))
print()

print(f"CF Recommendations for User {user_id}:")
print(get_cf_recommendations(user_id, n=5)[['title', 'genres', 'predicted_rating']].to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, 6), cv_results['test_rmse'], marker='o', label='RMSE', color='steelblue')
plt.plot(range(1, 6), cv_results['test_mae'],  marker='s', label='MAE',  color='coral')
plt.axhline(np.mean(cv_results['test_rmse']), linestyle='--', color='steelblue', alpha=0.5, label=f"Mean RMSE = {np.mean(cv_results['test_rmse']):.3f}")
plt.xlabel('Fold')
plt.ylabel('Error')
plt.title('SVD Model — 5-Fold Cross Validation')
plt.legend()
plt.tight_layout()
plt.show()